# Notebook 04 — Leakage-Safe Pre-Match Network Features

## Objective

This notebook converts the completed-match multilayer-network metrics created
in Notebook 03 into historical pre-match features.

The source network metrics describe what happened during each completed match.
They therefore cannot be used directly to predict that same match.

For every team, the workflow will:

1. order matches chronologically;
2. shift completed-match metrics by one fixture;
3. calculate rolling and expanding historical summaries;
4. create explicit history-count indicators;
5. reshape the team-level histories into home and away match features;
6. calculate home-minus-away feature differences;
7. validate that no current-match information enters the predictors.

The final output will contain one leakage-safe feature row per match.

In [1]:
# -------------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -------------------------------------------------------------------------
# DISPLAY SETTINGS
# -------------------------------------------------------------------------

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


# -------------------------------------------------------------------------
# REPRODUCIBILITY
# -------------------------------------------------------------------------

RANDOM_STATE = 42

print("Notebook 04 imports configured.")

Notebook 04 imports configured.


In [2]:
# -------------------------------------------------------------------------
# INPUT PATHS
# -------------------------------------------------------------------------

TEAM_MATCH_NETWORK_PATH = Path(
    "../data/processed/"
    "multilayer_team_match_metrics_laliga_2015_2016.parquet"
)

MATCHES_PATH = Path(
    "../data/processed/"
    "integrated_matches_laliga_2015_2016.parquet"
)


# -------------------------------------------------------------------------
# OUTPUT PATHS
# -------------------------------------------------------------------------

TEAM_HISTORY_OUTPUT_PATH = Path(
    "../data/processed/"
    "prematch_team_network_history_laliga_2015_2016.parquet"
)

MATCH_NETWORK_FEATURES_OUTPUT_PATH = Path(
    "../data/processed/"
    "prematch_match_network_features_laliga_2015_2016.parquet"
)


for output_path in [
    TEAM_HISTORY_OUTPUT_PATH,
    MATCH_NETWORK_FEATURES_OUTPUT_PATH,
]:
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


# -------------------------------------------------------------------------
# FILE VALIDATION
# -------------------------------------------------------------------------

required_input_paths = [
    TEAM_MATCH_NETWORK_PATH,
    MATCHES_PATH,
]

missing_input_paths = [
    path
    for path in required_input_paths
    if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join(
            str(path)
            for path in missing_input_paths
        )
    )

print("Notebook 04 paths configured.")
print(
    f"Team-match network input: "
    f"{TEAM_MATCH_NETWORK_PATH.resolve()}"
)
print(
    f"Integrated match input  : "
    f"{MATCHES_PATH.resolve()}"
)
print(
    f"Team-history output     : "
    f"{TEAM_HISTORY_OUTPUT_PATH.resolve()}"
)
print(
    f"Match-feature output    : "
    f"{MATCH_NETWORK_FEATURES_OUTPUT_PATH.resolve()}"
)

Notebook 04 paths configured.
Team-match network input: /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/multilayer_team_match_metrics_laliga_2015_2016.parquet
Integrated match input  : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/integrated_matches_laliga_2015_2016.parquet
Team-history output     : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_team_network_history_laliga_2015_2016.parquet
Match-feature output    : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_match_network_features_laliga_2015_2016.parquet


In [4]:
# -------------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------------

team_network_df = pd.read_parquet(
    TEAM_MATCH_NETWORK_PATH
)

matches_df = pd.read_parquet(
    MATCHES_PATH
)


# -------------------------------------------------------------------------
# BASIC NORMALIZATION
# -------------------------------------------------------------------------

team_network_df["match_date"] = pd.to_datetime(
    team_network_df["match_date"],
    errors="raise",
)

if "clean_date" in matches_df.columns:
    matches_df["match_date"] = pd.to_datetime(
        matches_df["clean_date"],
        errors="raise",
    )

elif "match_date" in matches_df.columns:
    matches_df["match_date"] = pd.to_datetime(
        matches_df["match_date"],
        errors="raise",
    )

else:
    raise ValueError(
        "The integrated match table contains neither "
        "'clean_date' nor 'match_date'."
    )


print("Source datasets loaded.")
print(
    f"Team-match network rows: "
    f"{len(team_network_df):,}"
)
print(
    f"Integrated match rows  : "
    f"{len(matches_df):,}"
)
print(
    f"Network matches        : "
    f"{team_network_df['match_id'].nunique():,}"
)
print(
    f"Integrated matches     : "
    f"{matches_df['match_id'].nunique():,}"
)

Source datasets loaded.
Team-match network rows: 760
Integrated match rows  : 380
Network matches        : 380
Integrated matches     : 380


In [5]:
# -------------------------------------------------------------------------
# SOURCE SCHEMA INSPECTION
# -------------------------------------------------------------------------

print("Team-match network columns:")
display(
    pd.DataFrame(
        {
            "column": team_network_df.columns,
            "dtype": [
                str(dtype)
                for dtype in team_network_df.dtypes
            ],
            "missing_count": [
                int(
                    team_network_df[column]
                    .isna()
                    .sum()
                )
                for column in team_network_df.columns
            ],
        }
    )
)

print("\nIntegrated match columns:")
display(
    pd.DataFrame(
        {
            "column": matches_df.columns,
            "dtype": [
                str(dtype)
                for dtype in matches_df.dtypes
            ],
            "missing_count": [
                int(
                    matches_df[column]
                    .isna()
                    .sum()
                )
                for column in matches_df.columns
            ],
        }
    )
)

Team-match network columns:


,column,dtype,missing_count
0,match_id,int64,0
1,match_date,datetime64[ns],0
2,home_team,object,0
3,away_team,object,0
4,home_score,int64,0
5,away_score,int64,0
6,layer,object,0
7,team,object,0
8,accumulated_eigenvector_centrality,float64,0
9,maximum_zone_centrality,float64,0



Integrated match columns:


,column,dtype,missing_count
0,match_id,int64,0
1,clean_date,datetime64[ns],0
2,home_team,string,0
3,away_team,string,0
4,home_score,int64,0
5,away_score,int64,0
6,implied_H,float64,0
7,implied_D,float64,0
8,implied_A,float64,0
9,bookie_margin,float64,0


In [6]:
# -------------------------------------------------------------------------
# REQUIRED TEAM-NETWORK COLUMNS
# -------------------------------------------------------------------------

required_team_network_columns = {
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "team",
    "layer",
    "is_home",
    "home_score",
    "away_score",
    "goals_for",
    "goals_against",
    "goal_difference",
    "completed_passes",
    "possession_losses",
    "possession_recoveries",
    "accumulated_eigenvector_centrality",
    "maximum_zone_centrality",
    "mean_zone_centrality",
    "mean_zone_leakage",
    "median_zone_leakage",
    "maximum_zone_leakage",
    "mean_zone_recovery",
    "median_zone_recovery",
    "maximum_zone_recovery",
    "switching_factor",
    "finite_switching_zones",
    "undefined_switching_zones",
}

missing_team_network_columns = (
    required_team_network_columns
    - set(team_network_df.columns)
)

if missing_team_network_columns:
    raise ValueError(
        "The team-match network table is missing columns:\n"
        f"{sorted(missing_team_network_columns)}"
    )


# -------------------------------------------------------------------------
# REQUIRED MATCH COLUMNS
# -------------------------------------------------------------------------

required_match_columns = {
    "match_id",
    "match_date",
    "home_team",
    "away_team",
}

missing_match_columns = (
    required_match_columns
    - set(matches_df.columns)
)

if missing_match_columns:
    raise ValueError(
        "The integrated match table is missing columns:\n"
        f"{sorted(missing_match_columns)}"
    )

print("Required source-column validation passed.")

Required source-column validation passed.


In [7]:
# -------------------------------------------------------------------------
# TEAM-MATCH STRUCTURAL VALIDATION
# -------------------------------------------------------------------------

assert len(team_network_df) == 760, (
    "Expected 760 team-match rows."
)

assert (
    team_network_df["match_id"].nunique()
    == 380
), (
    "Expected 380 unique matches."
)

duplicate_team_match_keys = (
    team_network_df[
        [
            "match_id",
            "team",
        ]
    ]
    .duplicated()
    .sum()
)

assert duplicate_team_match_keys == 0, (
    "Duplicate match-team keys detected."
)

team_rows_per_match = (
    team_network_df
    .groupby("match_id")
    .size()
)

assert team_rows_per_match.eq(2).all(), (
    "Every match must contain exactly two team rows."
)

home_rows_per_match = (
    team_network_df
    .groupby("match_id")[
        "is_home"
    ]
    .sum()
)

assert home_rows_per_match.eq(1).all(), (
    "Every match must contain exactly one home row."
)

assert set(
    team_network_df["layer"].unique()
) == {
    "home",
    "away",
}

assert (
    team_network_df["is_home"]
    .isin([0, 1])
    .all()
)

assert (
    team_network_df.loc[
        team_network_df["is_home"].eq(1),
        "layer",
    ]
    .eq("home")
    .all()
)

assert (
    team_network_df.loc[
        team_network_df["is_home"].eq(0),
        "layer",
    ]
    .eq("away")
    .all()
)

print("Team-match structural validation passed.")

Team-match structural validation passed.


In [8]:
# -------------------------------------------------------------------------
# HOME/AWAY TEAM IDENTITY VALIDATION
# -------------------------------------------------------------------------

home_identity_valid = (
    team_network_df.loc[
        team_network_df["is_home"].eq(1),
        "team",
    ]
    .eq(
        team_network_df.loc[
            team_network_df["is_home"].eq(1),
            "home_team",
        ]
    )
)

away_identity_valid = (
    team_network_df.loc[
        team_network_df["is_home"].eq(0),
        "team",
    ]
    .eq(
        team_network_df.loc[
            team_network_df["is_home"].eq(0),
            "away_team",
        ]
    )
)

assert home_identity_valid.all(), (
    "At least one home-layer team does not match home_team."
)

assert away_identity_valid.all(), (
    "At least one away-layer team does not match away_team."
)


# Check the match table against the network table.
network_match_identity_df = (
    team_network_df[
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
        ]
    ]
    .drop_duplicates()
)

assert len(network_match_identity_df) == 380

match_identity_validation_df = (
    network_match_identity_df
    .merge(
        matches_df[
            [
                "match_id",
                "match_date",
                "home_team",
                "away_team",
            ]
        ],
        on="match_id",
        how="outer",
        suffixes=(
            "_network",
            "_matches",
        ),
        indicator=True,
        validate="one_to_one",
    )
)

assert (
    match_identity_validation_df["_merge"]
    .eq("both")
    .all()
)

assert (
    match_identity_validation_df[
        "home_team_network"
    ]
    .eq(
        match_identity_validation_df[
            "home_team_matches"
        ]
    )
    .all()
)

assert (
    match_identity_validation_df[
        "away_team_network"
    ]
    .eq(
        match_identity_validation_df[
            "away_team_matches"
        ]
    )
    .all()
)

assert (
    match_identity_validation_df[
        "match_date_network"
    ]
    .eq(
        match_identity_validation_df[
            "match_date_matches"
        ]
    )
    .all()
)

print("Home/away identity validation passed.")

Home/away identity validation passed.


In [9]:
# -------------------------------------------------------------------------
# NETWORK METRICS TO BE HISTORICALLY AGGREGATED
# -------------------------------------------------------------------------

NETWORK_HISTORY_SOURCE_COLUMNS = [
    "completed_passes",
    "possession_losses",
    "possession_recoveries",
    "accumulated_eigenvector_centrality",
    "maximum_zone_centrality",
    "mean_zone_centrality",
    "mean_zone_leakage",
    "median_zone_leakage",
    "maximum_zone_leakage",
    "mean_zone_recovery",
    "median_zone_recovery",
    "maximum_zone_recovery",
    "switching_factor",
    "finite_switching_zones",
    "undefined_switching_zones",
]


# These outcome columns are retained only for validation and later target
# construction. They must never enter the historical feature calculations.
OUTCOME_COLUMNS = [
    "home_score",
    "away_score",
    "goals_for",
    "goals_against",
    "goal_difference",
]


missing_history_columns = [
    column
    for column in NETWORK_HISTORY_SOURCE_COLUMNS
    if column not in team_network_df.columns
]

if missing_history_columns:
    raise ValueError(
        "Historical source metrics are missing:\n"
        f"{missing_history_columns}"
    )


non_numeric_history_columns = [
    column
    for column in NETWORK_HISTORY_SOURCE_COLUMNS
    if not pd.api.types.is_numeric_dtype(
        team_network_df[column]
    )
]

if non_numeric_history_columns:
    raise TypeError(
        "Historical network metrics must be numeric:\n"
        f"{non_numeric_history_columns}"
    )

print(
    f"Historical source metrics selected: "
    f"{len(NETWORK_HISTORY_SOURCE_COLUMNS)}"
)

display(
    pd.DataFrame(
        {
            "historical_source_metric": (
                NETWORK_HISTORY_SOURCE_COLUMNS
            )
        }
    )
)

Historical source metrics selected: 15


,historical_source_metric
0,completed_passes
1,possession_losses
2,possession_recoveries
3,accumulated_eigenvector_centrality
4,maximum_zone_centrality
5,mean_zone_centrality
6,mean_zone_leakage
7,median_zone_leakage
8,maximum_zone_leakage
9,mean_zone_recovery


In [10]:
# -------------------------------------------------------------------------
# STRICT TEAM-MATCH CHRONOLOGY
# -------------------------------------------------------------------------

team_history_base_df = (
    team_network_df
    .sort_values(
        [
            "team",
            "match_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

team_history_base_df[
    "team_match_number"
] = (
    team_history_base_df
    .groupby("team")
    .cumcount()
    .add(1)
    .astype("int16")
)

team_history_base_df[
    "prior_matches_available"
] = (
    team_history_base_df[
        "team_match_number"
    ]
    .sub(1)
    .astype("int16")
)


# -------------------------------------------------------------------------
# CHRONOLOGY VALIDATION
# -------------------------------------------------------------------------

date_differences = (
    team_history_base_df
    .groupby("team")[
        "match_date"
    ]
    .diff()
)

assert (
    date_differences
    .dropna()
    .ge(pd.Timedelta(0))
    .all()
), (
    "At least one team's matches are not chronologically ordered."
)

assert (
    team_history_base_df
    .groupby("team")[
        "team_match_number"
    ]
    .min()
    .eq(1)
    .all()
)

assert (
    team_history_base_df[
        "prior_matches_available"
    ]
    .eq(
        team_history_base_df[
            "team_match_number"
        ]
        .sub(1)
    )
    .all()
)


team_match_counts_df = (
    team_history_base_df
    .groupby("team", as_index=False)
    .agg(
        matches_played=(
            "match_id",
            "size",
        ),
        first_match_date=(
            "match_date",
            "min",
        ),
        last_match_date=(
            "match_date",
            "max",
        ),
    )
    .sort_values("team")
    .reset_index(drop=True)
)

assert len(team_match_counts_df) == 20

assert (
    team_match_counts_df[
        "matches_played"
    ]
    .eq(38)
    .all()
), (
    "Every La Liga team should have 38 matches."
)

print("Strict team chronology constructed.")
print(
    f"Teams       : "
    f"{team_match_counts_df['team'].nunique():,}"
)
print(
    f"Rows        : "
    f"{len(team_history_base_df):,}"
)
print(
    f"Matches/team: "
    f"{team_match_counts_df['matches_played'].min()}–"
    f"{team_match_counts_df['matches_played'].max()}"
)

display(team_match_counts_df)

Strict team chronology constructed.
Teams       : 20
Rows        : 760
Matches/team: 38–38


,team,matches_played,first_match_date,last_match_date
0,Athletic Club,38,2015-08-23,2016-05-14
1,Atlético Madrid,38,2015-08-22,2016-05-14
2,Barcelona,38,2015-08-23,2016-05-14
3,Celta Vigo,38,2015-08-23,2016-05-14
4,Eibar,38,2015-08-24,2016-05-15
5,Espanyol,38,2015-08-22,2016-05-15
6,Getafe,38,2015-08-22,2016-05-15
7,Granada,38,2015-08-24,2016-05-14
8,Las Palmas,38,2015-08-22,2016-05-15
9,Levante UD,38,2015-08-23,2016-05-15


In [11]:
# -------------------------------------------------------------------------
# HISTORICAL WINDOW CONFIGURATION
# -------------------------------------------------------------------------

ROLLING_WINDOWS = [
    3,
    5,
    10,
]

MIN_PERIODS_BY_WINDOW = {
    3: 1,
    5: 1,
    10: 1,
}

print("Historical rolling windows configured.")
print(f"Rolling windows: {ROLLING_WINDOWS}")
print("Expanding season-to-date features will also be created.")

Historical rolling windows configured.
Rolling windows: [3, 5, 10]
Expanding season-to-date features will also be created.


In [12]:
# -------------------------------------------------------------------------
# ONE-MATCH SHIFT
# -------------------------------------------------------------------------

team_history_df = (
    team_history_base_df.copy()
)

shifted_source_columns = []

for metric_column in (
    NETWORK_HISTORY_SOURCE_COLUMNS
):
    shifted_column = (
        f"{metric_column}_lag1"
    )

    team_history_df[
        shifted_column
    ] = (
        team_history_df
        .groupby(
            "team",
            sort=False,
        )[metric_column]
        .shift(1)
    )

    shifted_source_columns.append(
        shifted_column
    )


# -------------------------------------------------------------------------
# CRITICAL LEAKAGE VALIDATION
# -------------------------------------------------------------------------

first_team_match_mask = (
    team_history_df[
        "team_match_number"
    ].eq(1)
)

assert team_history_df.loc[
    first_team_match_mask,
    shifted_source_columns,
].isna().all().all(), (
    "A team's first fixture contains historical values."
)

print("One-match historical shift completed.")
print(
    f"Shifted source columns created: "
    f"{len(shifted_source_columns)}"
)
print(
    f"First-match rows: "
    f"{first_team_match_mask.sum():,}"
)

display(
    team_history_df.loc[
        :,
        [
            "match_id",
            "match_date",
            "team",
            "team_match_number",
            "prior_matches_available",
        ]
        + shifted_source_columns[:5]
    ]
    .head(15)
)

One-match historical shift completed.
Shifted source columns created: 15
First-match rows: 20


,match_id,match_date,team,team_match_number,prior_matches_available,completed_passes_lag1,possession_losses_lag1,possession_recoveries_lag1,accumulated_eigenvector_centrality_lag1,maximum_zone_centrality_lag1
0,266236,2015-08-23,Athletic Club,1,0,NaN,NaN,NaN,NaN,NaN
1,3825578,2015-08-30,Athletic Club,2,1,215.0000,54.0000,53.0000,0.1254,0.0242
2,3825587,2015-09-13,Athletic Club,3,2,405.0000,64.0000,63.0000,0.7342,0.1383
3,3825595,2015-09-20,Athletic Club,4,3,390.0000,62.0000,62.0000,0.6375,0.0751
4,3825602,2015-09-23,Athletic Club,5,4,341.0000,60.0000,60.0000,0.6518,0.1173
5,3825609,2015-09-27,Athletic Club,6,5,369.0000,61.0000,60.0000,0.2144,0.0389
6,3825622,2015-10-04,Athletic Club,7,6,331.0000,69.0000,70.0000,0.5996,0.1211
7,3825630,2015-10-18,Athletic Club,8,7,383.0000,60.0000,61.0000,0.4482,0.1092
8,3825641,2015-10-26,Athletic Club,9,8,289.0000,63.0000,62.0000,0.1538,0.0217
9,3825650,2015-11-01,Athletic Club,10,9,383.0000,66.0000,67.0000,0.4385,0.0580


In [13]:
# -------------------------------------------------------------------------
# ROLLING HISTORICAL NETWORK MEANS
# -------------------------------------------------------------------------

rolling_feature_columns = []

for window_size in ROLLING_WINDOWS:

    for metric_column in (
        NETWORK_HISTORY_SOURCE_COLUMNS
    ):
        lagged_column = (
            f"{metric_column}_lag1"
        )

        feature_column = (
            f"{metric_column}"
            f"_roll{window_size}_mean"
        )

        team_history_df[
            feature_column
        ] = (
            team_history_df
            .groupby(
                "team",
                sort=False,
            )[lagged_column]
            .transform(
                lambda series: (
                    series
                    .rolling(
                        window=window_size,
                        min_periods=(
                            MIN_PERIODS_BY_WINDOW[
                                window_size
                            ]
                        ),
                    )
                    .mean()
                )
            )
        )

        rolling_feature_columns.append(
            feature_column
        )

print("Rolling historical network means created.")
print(
    f"Rolling feature columns: "
    f"{len(rolling_feature_columns):,}"
)

Rolling historical network means created.
Rolling feature columns: 45


In [14]:
# -------------------------------------------------------------------------
# EXPANDING HISTORICAL NETWORK MEANS
# -------------------------------------------------------------------------

expanding_feature_columns = []

for metric_column in (
    NETWORK_HISTORY_SOURCE_COLUMNS
):
    lagged_column = (
        f"{metric_column}_lag1"
    )

    feature_column = (
        f"{metric_column}"
        "_expanding_mean"
    )

    team_history_df[
        feature_column
    ] = (
        team_history_df
        .groupby(
            "team",
            sort=False,
        )[lagged_column]
        .transform(
            lambda series: (
                series
                .expanding(
                    min_periods=1
                )
                .mean()
            )
        )
    )

    expanding_feature_columns.append(
        feature_column
    )

print("Expanding historical features created.")
print(
    f"Expanding feature columns: "
    f"{len(expanding_feature_columns):,}"
)

Expanding historical features created.
Expanding feature columns: 15


In [15]:
# -------------------------------------------------------------------------
# MANUAL LEAKAGE-SAFETY CHECK
# -------------------------------------------------------------------------

sample_team = (
    team_history_df["team"]
    .sort_values()
    .iloc[0]
)

sample_team_history_df = (
    team_history_df.loc[
        team_history_df["team"]
        .eq(sample_team)
    ]
    .sort_values(
        [
            "match_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)

sample_metric = (
    "accumulated_eigenvector_centrality"
)

manual_check_rows = []

for row_index in range(
    min(
        8,
        len(sample_team_history_df),
    )
):
    current_row = (
        sample_team_history_df
        .iloc[row_index]
    )

    previous_rows_df = (
        sample_team_history_df
        .iloc[
            max(
                0,
                row_index - 3,
            ):
            row_index
        ]
    )

    if previous_rows_df.empty:
        manual_roll3 = np.nan

    else:
        manual_roll3 = (
            previous_rows_df[
                sample_metric
            ]
            .mean()
        )

    stored_roll3 = (
        current_row[
            f"{sample_metric}"
            "_roll3_mean"
        ]
    )

    if pd.isna(manual_roll3):
        assert pd.isna(stored_roll3)

    else:
        assert np.isclose(
            manual_roll3,
            stored_roll3,
            atol=1e-12,
        )

    manual_check_rows.append(
        {
            "team": sample_team,
            "match_id": int(
                current_row[
                    "match_id"
                ]
            ),
            "team_match_number": int(
                current_row[
                    "team_match_number"
                ]
            ),
            "manual_previous_3_mean": (
                manual_roll3
            ),
            "stored_roll3_mean": (
                stored_roll3
            ),
        }
    )

manual_check_df = pd.DataFrame(
    manual_check_rows
)

print(
    "Manual rolling-history validation passed."
)

display(manual_check_df)

Manual rolling-history validation passed.


,team,match_id,team_match_number,manual_previous_3_mean,stored_roll3_mean
0,Athletic Club,266236,1,NaN,NaN
1,Athletic Club,3825578,2,0.1254,0.1254
2,Athletic Club,3825587,3,0.4298,0.4298
3,Athletic Club,3825595,4,0.4990,0.4990
4,Athletic Club,3825602,5,0.6745,0.6745
5,Athletic Club,3825609,6,0.5012,0.5012
6,Athletic Club,3825622,7,0.4886,0.4886
7,Athletic Club,3825630,8,0.4207,0.4207


In [16]:
# -------------------------------------------------------------------------
# EARLY-SEASON FEATURE VALIDATION
# -------------------------------------------------------------------------

historical_feature_columns = (
    rolling_feature_columns
    + expanding_feature_columns
)

first_match_rows_df = (
    team_history_df.loc[
        team_history_df[
            "team_match_number"
        ].eq(1)
    ]
)

second_match_rows_df = (
    team_history_df.loc[
        team_history_df[
            "team_match_number"
        ].eq(2)
    ]
)

assert len(first_match_rows_df) == 20
assert len(second_match_rows_df) == 20

assert first_match_rows_df[
    historical_feature_columns
].isna().all().all(), (
    "First team fixtures should contain "
    "no network history."
)

# By the second fixture, each team has one prior completed match.
assert second_match_rows_df[
    historical_feature_columns
].notna().all().all(), (
    "Second fixtures should have one prior "
    "match available for all historical means."
)

print("Early-season history validation passed.")
print(
    "First fixtures with no history: "
    f"{len(first_match_rows_df)}"
)
print(
    "Second fixtures with one-match history: "
    f"{len(second_match_rows_df)}"
)

Early-season history validation passed.
First fixtures with no history: 20
Second fixtures with one-match history: 20


In [17]:
# -------------------------------------------------------------------------
# CONSTRUCT TEAM-LEVEL PRE-MATCH HISTORY TABLE
# -------------------------------------------------------------------------

team_history_identifier_columns = [
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "team",
    "layer",
    "is_home",
    "team_match_number",
    "prior_matches_available",
]

team_history_feature_columns = (
    rolling_feature_columns
    + expanding_feature_columns
)

prematch_team_history_df = (
    team_history_df[
        team_history_identifier_columns
        + team_history_feature_columns
    ]
    .copy()
)


# -------------------------------------------------------------------------
# SAME-MATCH NETWORK LEAKAGE CHECK
# -------------------------------------------------------------------------

for raw_metric_column in (
    NETWORK_HISTORY_SOURCE_COLUMNS
):
    assert (
        raw_metric_column
        not in prematch_team_history_df.columns
    )

for outcome_column in OUTCOME_COLUMNS:
    assert (
        outcome_column
        not in prematch_team_history_df.columns
    )

assert len(
    prematch_team_history_df
) == 760

assert not prematch_team_history_df[
    [
        "match_id",
        "team",
    ]
].duplicated().any()

print(
    "Leakage-safe team-history table constructed."
)
print(
    f"Rows    : "
    f"{len(prematch_team_history_df):,}"
)
print(
    f"Features: "
    f"{len(team_history_feature_columns):,}"
)

display(
    prematch_team_history_df.head()
)

Leakage-safe team-history table constructed.
Rows    : 760
Features: 60


,match_id,match_date,home_team,away_team,team,layer,is_home,team_match_number,prior_matches_available,completed_passes_roll3_mean,possession_losses_roll3_mean,possession_recoveries_roll3_mean,accumulated_eigenvector_centrality_roll3_mean,maximum_zone_centrality_roll3_mean,mean_zone_centrality_roll3_mean,mean_zone_leakage_roll3_mean,median_zone_leakage_roll3_mean,maximum_zone_leakage_roll3_mean,mean_zone_recovery_roll3_mean,median_zone_recovery_roll3_mean,maximum_zone_recovery_roll3_mean,switching_factor_roll3_mean,finite_switching_zones_roll3_mean,undefined_switching_zones_roll3_mean,completed_passes_roll5_mean,possession_losses_roll5_mean,possession_recoveries_roll5_mean,accumulated_eigenvector_centrality_roll5_mean,maximum_zone_centrality_roll5_mean,mean_zone_centrality_roll5_mean,mean_zone_leakage_roll5_mean,median_zone_leakage_roll5_mean,maximum_zone_leakage_roll5_mean,mean_zone_recovery_roll5_mean,median_zone_recovery_roll5_mean,maximum_zone_recovery_roll5_mean,switching_factor_roll5_mean,finite_switching_zones_roll5_mean,undefined_switching_zones_roll5_mean,completed_passes_roll10_mean,possession_losses_roll10_mean,possession_recoveries_roll10_mean,accumulated_eigenvector_centrality_roll10_mean,maximum_zone_centrality_roll10_mean,mean_zone_centrality_roll10_mean,mean_zone_leakage_roll10_mean,median_zone_leakage_roll10_mean,maximum_zone_leakage_roll10_mean,mean_zone_recovery_roll10_mean,median_zone_recovery_roll10_mean,maximum_zone_recovery_roll10_mean,switching_factor_roll10_mean,finite_switching_zones_roll10_mean,undefined_switching_zones_roll10_mean,completed_passes_expanding_mean,possession_losses_expanding_mean,possession_recoveries_expanding_mean,accumulated_eigenvector_centrality_expanding_mean,maximum_zone_centrality_expanding_mean,mean_zone_centrality_expanding_mean,mean_zone_leakage_expanding_mean,median_zone_leakage_expanding_mean,maximum_zone_leakage_expanding_mean,mean_zone_recovery_expanding_mean,median_zone_recovery_expanding_mean,maximum_zone_recovery_expanding_mean,switching_factor_expanding_mean,finite_switching_zones_expanding_mean,undefined_switching_zones_expanding_mean
0,266236,2015-08-23,Athletic Club,Barcelona,Athletic Club,home,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3825578,2015-08-30,Eibar,Athletic Club,Athletic Club,away,0,2,1,215.0000,54.0000,53.0000,0.1254,0.0242,0.0063,0.2217,0.2248,1.0000,0.1895,0.1750,0.6667,6.1207,20.0000,0.0000,215.0000,54.0000,53.0000,0.1254,0.0242,0.0063,0.2217,0.2248,1.0000,0.1895,0.1750,0.6667,6.1207,20.0000,0.0000,215.0000,54.0000,53.0000,0.1254,0.0242,0.0063,0.2217,0.2248,1.0000,0.1895,0.1750,0.6667,6.1207,20.0000,0.0000,215.0000,54.0000,53.0000,0.1254,0.0242,0.0063,0.2217,0.2248,1.0000,0.1895,0.1750,0.6667,6.1207,20.0000,0.0000
2,3825587,2015-09-13,Athletic Club,Getafe,Athletic Club,home,1,3,2,310.0000,59.0000,58.0000,0.4298,0.0812,0.0215,0.2156,0.1693,0.9444,0.1760,0.1419,0.6333,5.7247,20.0000,0.0000,310.0000,59.0000,58.0000,0.4298,0.0812,0.0215,0.2156,0.1693,0.9444,0.1760,0.1419,0.6333,5.7247,20.0000,0.0000,310.0000,59.0000,58.0000,0.4298,0.0812,0.0215,0.2156,0.1693,0.9444,0.1760,0.1419,0.6333,5.7247,20.0000,0.0000,310.0000,59.0000,58.0000,0.4298,0.0812,0.0215,0.2156,0.1693,0.9444,0.1760,0.1419,0.6333,5.7247,20.0000,0.0000
3,3825595,2015-09-20,Villarreal,Athletic Club,Athletic Club,away,0,4,3,336.6667,60.0000,59.3333,0.4990,0.0792,0.0250,0.1927,0.1454,0.8272,0.1797,0.1252,0.7556,5.1052,19.6667,0.3333,336.6667,60.0000,59.3333,0.4990,0.0792,0.0250,0.1927,0.1454,0.8272,0.1797,0.1252,0.7556,5.1052,19.6667,0.3333,336.6667,60.0000,59.3333,0.4990,0.0792,0.0250,0.1927,0.1454,0.8272,0.1797,0.1252,0.7556,5.1052,19.6667,0.3333,336.6667,60.0000,59.3333,0.4990,0.0792,0.0250,0.1927,0.1454,0.8272,0.1797,0.1252,0.7556,5.1052,19.6667,0.3333
4,3825602,2015-09-23,Athletic Club

In [18]:
# -------------------------------------------------------------------------
# HOME TEAM HISTORY TABLE
# -------------------------------------------------------------------------

home_team_history_df = (
    prematch_team_history_df.loc[
        prematch_team_history_df[
            "is_home"
        ].eq(1)
    ]
    .copy()
)

assert len(home_team_history_df) == 380

assert (
    home_team_history_df["team"]
    == home_team_history_df[
        "home_team"
    ]
).all()


home_keep_columns = [
    "match_id",
    "match_date",
    "team",
    "team_match_number",
    "prior_matches_available",
] + team_history_feature_columns

home_team_history_df = (
    home_team_history_df[
        home_keep_columns
    ]
    .rename(
        columns={
            "team": "home_team",
            "team_match_number": (
                "home_team_match_number"
            ),
            "prior_matches_available": (
                "home_prior_matches_available"
            ),
            **{
                column: f"home_{column}"
                for column
                in team_history_feature_columns
            },
        }
    )
)

print("Home-team history table created.")
print(
    f"Rows: {len(home_team_history_df):,}"
)

Home-team history table created.
Rows: 380


In [19]:
# -------------------------------------------------------------------------
# AWAY TEAM HISTORY TABLE
# -------------------------------------------------------------------------

away_team_history_df = (
    prematch_team_history_df.loc[
        prematch_team_history_df[
            "is_home"
        ].eq(0)
    ]
    .copy()
)

assert len(away_team_history_df) == 380

assert (
    away_team_history_df["team"]
    == away_team_history_df[
        "away_team"
    ]
).all()


away_keep_columns = [
    "match_id",
    "match_date",
    "team",
    "team_match_number",
    "prior_matches_available",
] + team_history_feature_columns

away_team_history_df = (
    away_team_history_df[
        away_keep_columns
    ]
    .rename(
        columns={
            "team": "away_team",
            "team_match_number": (
                "away_team_match_number"
            ),
            "prior_matches_available": (
                "away_prior_matches_available"
            ),
            **{
                column: f"away_{column}"
                for column
                in team_history_feature_columns
            },
        }
    )
)

print("Away-team history table created.")
print(
    f"Rows: {len(away_team_history_df):,}"
)

Away-team history table created.
Rows: 380


In [20]:
# -------------------------------------------------------------------------
# MATCH-LEVEL PRE-MATCH NETWORK FEATURES
# -------------------------------------------------------------------------

match_identity_df = (
    matches_df[
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
        ]
    ]
    .drop_duplicates()
    .copy()
)

assert len(match_identity_df) == 380


prematch_match_network_df = (
    match_identity_df
    .merge(
        home_team_history_df,
        on=[
            "match_id",
            "match_date",
            "home_team",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        away_team_history_df,
        on=[
            "match_id",
            "match_date",
            "away_team",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(
    prematch_match_network_df
) == 380

assert not prematch_match_network_df[
    "match_id"
].duplicated().any()

print(
    "Match-level pre-match network table created."
)
print(
    f"Rows: "
    f"{len(prematch_match_network_df):,}"
)
print(
    f"Columns: "
    f"{prematch_match_network_df.shape[1]:,}"
)

Match-level pre-match network table created.
Rows: 380
Columns: 128


In [21]:
# -------------------------------------------------------------------------
# HOME-MINUS-AWAY NETWORK DIFFERENCES
# -------------------------------------------------------------------------

difference_feature_columns = []

for base_feature in (
    team_history_feature_columns
):
    home_column = (
        f"home_{base_feature}"
    )

    away_column = (
        f"away_{base_feature}"
    )

    difference_column = (
        f"diff_{base_feature}"
    )

    prematch_match_network_df[
        difference_column
    ] = (
        prematch_match_network_df[
            home_column
        ]
        - prematch_match_network_df[
            away_column
        ]
    )

    difference_feature_columns.append(
        difference_column
    )


prematch_match_network_df[
    "diff_prior_matches_available"
] = (
    prematch_match_network_df[
        "home_prior_matches_available"
    ]
    - prematch_match_network_df[
        "away_prior_matches_available"
    ]
)

print(
    "Home-minus-away difference features created."
)
print(
    f"Difference features: "
    f"{len(difference_feature_columns):,}"
)

Home-minus-away difference features created.
Difference features: 60


In [22]:
# -------------------------------------------------------------------------
# EARLY-SEASON MISSING-HISTORY DIAGNOSTICS
# -------------------------------------------------------------------------

home_feature_columns = [
    f"home_{column}"
    for column
    in team_history_feature_columns
]

away_feature_columns = [
    f"away_{column}"
    for column
    in team_history_feature_columns
]

complete_history_mask = (
    prematch_match_network_df[
        home_feature_columns
        + away_feature_columns
    ]
    .notna()
    .all(axis=1)
)

print(
    f"Matches with complete network history: "
    f"{complete_history_mask.sum():,}"
)

print(
    f"Matches containing at least one "
    f"missing network-history value: "
    f"{(~complete_history_mask).sum():,}"
)


history_distribution_df = (
    prematch_match_network_df[
        [
            "home_prior_matches_available",
            "away_prior_matches_available",
        ]
    ]
    .describe()
)

display(history_distribution_df)


first_round_like_matches_df = (
    prematch_match_network_df.loc[
        prematch_match_network_df[
            "home_prior_matches_available"
        ].eq(0)
        | prematch_match_network_df[
            "away_prior_matches_available"
        ].eq(0),
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "home_prior_matches_available",
            "away_prior_matches_available",
        ],
    ]
)

display(
    first_round_like_matches_df
    .sort_values(
        [
            "match_date",
            "match_id",
        ]
    )
)

Matches with complete network history: 370
Matches containing at least one missing network-history value: 10


,home_prior_matches_available,away_prior_matches_available
count,380.0000,380.0000
mean,18.4974,18.5026
std,10.9794,10.9813
min,0.0000,0.0000
25%,9.0000,9.0000
50%,18.5000,18.5000
75%,28.0000,28.0000
max,37.0000,37.0000


,match_id,match_date,home_team,away_team,home_prior_matches_available,away_prior_matches_available
0,3825562,2015-08-21,Málaga,Sevilla,0,0
1,3825563,2015-08-22,Atlético Madrid,Las Palmas,0,0
2,3825564,2015-08-22,RC Deportivo La Coruña,Real Sociedad,0,0
3,3825565,2015-08-22,Espanyol,Getafe,0,0
4,3825566,2015-08-22,Rayo Vallecano,Valencia,0,0
5,266236,2015-08-23,Athletic Club,Barcelona,0,0
6,3825567,2015-08-23,Sporting Gijón,Real Madrid,0,0
7,3825568,2015-08-23,Real Betis,Villarreal,0,0
8,3825569,2015-08-23,Levante UD,Celta Vigo,0,0
9,3825570,2015-08-24,Granada,Eibar,0,0


In [23]:
# -------------------------------------------------------------------------
# FINAL LEAKAGE AUDIT
# -------------------------------------------------------------------------

forbidden_exact_columns = set(
    NETWORK_HISTORY_SOURCE_COLUMNS
    + OUTCOME_COLUMNS
    + shifted_source_columns
)

present_forbidden_columns = (
    forbidden_exact_columns
    & set(
        prematch_match_network_df.columns
    )
)

assert not present_forbidden_columns, (
    "Potential same-match leakage columns detected:\n"
    f"{sorted(present_forbidden_columns)}"
)


# Every historical feature name must explicitly describe a
# rolling or expanding statistic.
unexpected_network_feature_columns = [
    column
    for column
    in prematch_match_network_df.columns
    if (
        any(
            metric in column
            for metric
            in NETWORK_HISTORY_SOURCE_COLUMNS
        )
        and (
            "_roll" not in column
            and "_expanding_" not in column
        )
    )
]

assert not unexpected_network_feature_columns, (
    "Unexpected non-historical network columns detected:\n"
    f"{unexpected_network_feature_columns}"
)


# The first fixture for each team must have no derived history.
first_match_team_rows = (
    prematch_team_history_df.loc[
        prematch_team_history_df[
            "prior_matches_available"
        ].eq(0)
    ]
)

assert len(first_match_team_rows) == 20

assert first_match_team_rows[
    team_history_feature_columns
].isna().all().all()

print("Final leakage audit passed.")
print(
    "No current-match network metrics or "
    "outcome variables are present."
)

Final leakage audit passed.
No current-match network metrics or outcome variables are present.


In [24]:
# -------------------------------------------------------------------------
# FINAL MATCH-LEVEL STRUCTURAL VALIDATION
# -------------------------------------------------------------------------

assert len(
    prematch_match_network_df
) == 380

assert (
    prematch_match_network_df[
        "match_id"
    ].nunique()
    == 380
)

assert not prematch_match_network_df[
    "match_id"
].duplicated().any()

assert (
    prematch_match_network_df[
        "home_team"
    ].notna().all()
)

assert (
    prematch_match_network_df[
        "away_team"
    ].notna().all()
)

assert (
    prematch_match_network_df[
        "home_prior_matches_available"
    ].between(
        0,
        37,
    ).all()
)

assert (
    prematch_match_network_df[
        "away_prior_matches_available"
    ].between(
        0,
        37,
    ).all()
)

assert (
    prematch_team_history_df[
        "prior_matches_available"
    ].max()
    == 37
)

print(
    "Final match-level structural validation passed."
)
print(
    f"Matches : "
    f"{len(prematch_match_network_df):,}"
)
print(
    f"Columns : "
    f"{prematch_match_network_df.shape[1]:,}"
)

Final match-level structural validation passed.
Matches : 380
Columns : 189


In [25]:
# -------------------------------------------------------------------------
# SELECTED FEATURE DISTRIBUTIONS
# -------------------------------------------------------------------------

inspection_columns = [
    "home_accumulated_eigenvector_centrality_roll5_mean",
    "away_accumulated_eigenvector_centrality_roll5_mean",
    "diff_accumulated_eigenvector_centrality_roll5_mean",
    "home_mean_zone_leakage_roll5_mean",
    "away_mean_zone_leakage_roll5_mean",
    "diff_mean_zone_leakage_roll5_mean",
    "home_mean_zone_recovery_roll5_mean",
    "away_mean_zone_recovery_roll5_mean",
    "diff_mean_zone_recovery_roll5_mean",
    "home_switching_factor_roll5_mean",
    "away_switching_factor_roll5_mean",
    "diff_switching_factor_roll5_mean",
]

display(
    prematch_match_network_df[
        inspection_columns
    ]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
home_accumulated_eigenvector_centrality_roll5_mean,370.0000,0.4945,0.1783,0.1001,0.3570,0.4691,0.6009,0.9422
away_accumulated_eigenvector_centrality_roll5_mean,370.0000,0.5055,0.1790,0.0578,0.3703,0.4842,0.6224,0.9239
diff_accumulated_eigenvector_centrality_roll5_mean,370.0000,-0.0110,0.2476,-0.6645,-0.1798,-0.0175,0.1752,0.6753
home_mean_zone_leakage_roll5_mean,370.0000,0.2191,0.0469,0.0921,0.1909,0.2203,0.2500,0.4698
away_mean_zone_leakage_roll5_mean,370.0000,0.2161,0.0455,0.1066,0.1858,0.2196,0.2440,0.4373
diff_mean_zone_leakage_roll5_mean,370.0000,0.0030,0.0693,-0.2264,-0.0421,0.0014,0.0492,0.2681
home_mean_zone_recovery_roll5_mean,370.0000,0.1635,0.0283,0.0644,0.1461,0.1643,0.1810,0.2624
away_mean_zone_recovery_roll5_mean,370.0000,0.1633,0.0278,0.0933,0.1443,0.1653,0.1815,0.2606
diff_mean_zone_recovery_roll5_mean,370.0000,0.0002,0.0392,-0.1224,-0.0261,0.0008,0.0264,0.1098
home_switching_factor_roll5_mean,370.0000,5.9708,1.7046,2.7297,4.8978,5.8857,6.7736,15.4830


In [26]:
# -------------------------------------------------------------------------
# EXPORT LEAKAGE-SAFE NETWORK FEATURES
# -------------------------------------------------------------------------

prematch_team_history_df.to_parquet(
    TEAM_HISTORY_OUTPUT_PATH,
    index=False,
)

prematch_match_network_df.to_parquet(
    MATCH_NETWORK_FEATURES_OUTPUT_PATH,
    index=False,
)

print("Notebook 04 outputs exported.")
print(
    f"Team-history rows: "
    f"{len(prematch_team_history_df):,}"
)
print(
    f"Match-feature rows: "
    f"{len(prematch_match_network_df):,}"
)

print(
    f"\nTeam-history file:\n"
    f"{TEAM_HISTORY_OUTPUT_PATH.resolve()}"
)

print(
    f"\nMatch-feature file:\n"
    f"{MATCH_NETWORK_FEATURES_OUTPUT_PATH.resolve()}"
)

Notebook 04 outputs exported.
Team-history rows: 760
Match-feature rows: 380

Team-history file:
/home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_team_network_history_laliga_2015_2016.parquet

Match-feature file:
/home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_match_network_features_laliga_2015_2016.parquet


In [27]:
# -------------------------------------------------------------------------
# RELOAD VALIDATION
# -------------------------------------------------------------------------

reloaded_team_history_df = (
    pd.read_parquet(
        TEAM_HISTORY_OUTPUT_PATH
    )
)

reloaded_match_network_df = (
    pd.read_parquet(
        MATCH_NETWORK_FEATURES_OUTPUT_PATH
    )
)

assert len(
    reloaded_team_history_df
) == 760

assert len(
    reloaded_match_network_df
) == 380

assert not reloaded_team_history_df[
    [
        "match_id",
        "team",
    ]
].duplicated().any()

assert not reloaded_match_network_df[
    "match_id"
].duplicated().any()

assert set(
    reloaded_match_network_df[
        "match_id"
    ]
) == set(
    matches_df["match_id"]
)

print("Notebook 04 reload validation passed.")
print(
    f"Reloaded team-history rows: "
    f"{len(reloaded_team_history_df):,}"
)
print(
    f"Reloaded match-feature rows: "
    f"{len(reloaded_match_network_df):,}"
)

Notebook 04 reload validation passed.
Reloaded team-history rows: 760
Reloaded match-feature rows: 380


# Notebook 04 Summary — Leakage-Safe Pre-Match Network Features

## Objective

This notebook transformed the completed-match multilayer-network metrics from
Notebook 03 into historical predictors that are safe to use before kickoff.

The raw network values from a match were never used as predictors for that same
match.

---

## Temporal construction

For every team, fixtures were ordered using:

```text
team
match_date
match_id